In [1]:
import torch
print(torch.__version__)

print(sys.version)
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.11.0+cu128
3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:37:13) [MSC v.1942 64 bit (AMD64)]
2.11.0+cu128
True
NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
# %pip install pandas
# %pip install numpy
# %pip install -U scikit-learn
# %pip install transformers
# %pip install emoji
# %pip install requests

In [3]:
import pandas as pd
import numpy as np
import emoji
import requests
import random

import json

# SKLearn import
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report # For accuracy score
from sklearn.preprocessing import MultiLabelBinarizer # Convert multiple emotion into format so model can train



In [4]:
# Transformer
import transformers
from transformers import AutoTokenizer
from transformers import XLMRobertaModel # AI Model download

# Import/install training model (XLM-RoBERTa)
model_name = "FacebookAI/xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = XLMRobertaModel.from_pretrained(model_name)

C:\Users\ASUS\.conda\envs\lily-ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 629.55it/s]
[transformers] XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# Testing purpose
text = 'I am really happy today!'
inputs = tokenizer(text, return_tensors='pt')
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

torch.Size([1, 8, 768])


In [6]:
df = pd.read_csv('emotion_data.csv')

In [7]:
df.sample(10)

,message,Happy,Sad,Angry,Frustrated,Bored,Excited,Jealous,Insecure,Relieved,Neutral
966,"I was talking with a friend about puberty, and...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1662,Meanwhile I cannot just cancel being a parent ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1211,"But if I were better, maybe I could have chang...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1734,That is probably what I am missing.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
361,It's been years,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1269,I told my friend about an idea I had for a pro...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
611,I just want something interesting 😭,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
207,I want to buy something,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
925,Is there a specific age when puberty is suppos...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44,Thats literally what im doing lol,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Cleaning and data confirming

In [8]:
df.duplicated().sum()

np.int64(101)

In [9]:
df.isna().sum()

message          0
Happy         1737
Sad           1737
Angry         1737
Frustrated    1737
Bored         1737
Excited       1737
Jealous       1737
Insecure      1737
Relieved      1737
Neutral       1737
dtype: int64

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1737 entries, 0 to 1736
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   message     1737 non-null   str    
 1   Happy       0 non-null      float64
 2   Sad         0 non-null      float64
 3   Angry       0 non-null      float64
 4   Frustrated  0 non-null      float64
 5   Bored       0 non-null      float64
 6   Excited     0 non-null      float64
 7   Jealous     0 non-null      float64
 8   Insecure    0 non-null      float64
 9   Relieved    0 non-null      float64
 10  Neutral     0 non-null      float64
dtypes: float64(10), str(1)
memory usage: 149.4 KB


In [11]:
df.shape

(1737, 11)

In [12]:
df.nunique()

message       1636
Happy            0
Sad              0
Angry            0
Frustrated       0
Bored            0
Excited          0
Jealous          0
Insecure         0
Relieved         0
Neutral          0
dtype: int64

In [13]:
df.columns

Index(['message', 'Happy', 'Sad', 'Angry', 'Frustrated', 'Bored', 'Excited',
       'Jealous', 'Insecure', 'Relieved', 'Neutral'],
      dtype='str')

In [14]:
df['message'] = df.message.str.strip()

In [15]:
line_break = df['message'].str.contains('\n', regex=False).sum()
print(f"Total message with line break: {line_break}")

Total message with line break: 8


In [16]:
emoji_count_message = df.message.apply(emoji.emoji_count).gt(0).sum()
print(f"Total message containing emoji: {emoji_count_message}")

emoji_count = df.message.apply(emoji.emoji_count).sum()
print(f"Total emoji count: {emoji_count}")

Total message containing emoji: 266
Total emoji count: 448


# Making new dataframe, text only

In [17]:
df_text = df.copy()

In [18]:
df_text['message'] = df_text.message.apply(
    lambda x: emoji.replace_emoji(x, replace='')
)

In [19]:
df_text.message.apply(emoji.emoji_count).sum()

np.int64(0)

In [20]:
df_text.columns

Index(['message', 'Happy', 'Sad', 'Angry', 'Frustrated', 'Bored', 'Excited',
       'Jealous', 'Insecure', 'Relieved', 'Neutral'],
      dtype='str')

## Insert local LLM model to label emotion for each message

In [21]:
df_text.head()

,message,Happy,Sad,Angry,Frustrated,Bored,Excited,Jealous,Insecure,Relieved,Neutral
0,"Hi, Lily. Do you know who am I?",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,You know that? I wanted to test something.\naA...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Hi Lily. How do you do?,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Only sad things ever happened today... I watch...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Ah well, I've never told her my feelings anywa...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
emotion_names = [
    "Happy", "Sad", "Angry", "Frustrated", "Bored",
    "Excited", "Jealous", "Insecure", "Relieved", "Neutral"
]

unlabeled = df_text[
    df_text[emotion_names].isna().all(axis=1)
]

sample = unlabeled.sample(n=min(100, len(unlabeled)))

count = 0
success_count = 0
failed_rows = []

for index, message in sample["message"].items():

    prompt = f"""
    You are an emotion annotation assistant.

    Analyze the emotional signals expressed or reasonably implied
    by the following message. Base your judgment on the message
    itself, not assumptions about the speaker.

    EMOTION DEFINITIONS:

    Happy:
    Happiness, contentment, pleasure, or satisfaction.
    Includes feeling glad, cheerful, or pleased with an outcome.

    Sad:
    Sorrow, emotional pain, grief, loneliness, disappointment,
    or feeling down.

    Angry:
    Strong displeasure, hostility, or resentment toward someone
    or something perceived as wrong, unfair, or offensive.

    Frustrated:
    Annoyance or distress caused by obstacles, difficulties,
    unmet goals, or feeling unable to make progress.

    Bored:
    Lack of interest, stimulation, or engagement.
    Includes feeling that something is dull or repetitive.

    Excited:
    High-energy anticipation, enthusiasm, or eagerness about
    something enjoyable or significant.

    Jealous:
    Distress, resentment, or threat related to someone else
    receiving attention, affection, opportunities, or possessions
    that the speaker wants or values.

    Insecure:
    Doubting one's own worth, ability, attractiveness, or acceptance.
    Includes fear of rejection, inadequacy, or being inferior.

    Relieved:
    A reduction in worry, fear, tension, or pressure after a
    problem is resolved or a feared outcome is avoided.

    Neutral:
    No clearly detectable emotional tone. The message is primarily
    factual, informational, or emotionally unmarked.

    IMPORTANT DISTINCTIONS:

    - Angry focuses on displeasure or perceived wrongdoing.
      Frustrated focuses on obstacles or unmet goals.
      Both may be present.
    - Sad focuses on emotional pain or loss.
      Insecure focuses on self-doubt or fear of rejection.
      Both may be present.
    - Happy is general positive emotion.
      Excited includes stronger enthusiasm or anticipation.
    - Do not infer an emotion from the topic alone.
      For example, mentioning a relationship does not automatically
      mean jealousy or insecurity.
    - Short messages are not automatically neutral.
      Judge the emotional evidence available in the message.
    - Mixed emotions are possible. A person can feel happy and
      jealous, or relieved and sad, at the same time.
    - Score each emotion independently. Do not force one emotion
      to be the dominant emotion.

    INTENSITY SCALE:

    Give each emotion an independent intensity score from 0 to 100.

    0 = no detectable evidence of the emotion
    1-20 = very weak
    21-40 = mild
    41-60 = moderate
    61-80 = strong
    81-99 = very strong
    100 = extremely strong

    These are emotion intensity scores, NOT probabilities.
    Scores do not need to add up to 100.
    Use 0 when there is no reasonable evidence for an emotion.
    Do not assign high scores based only on speculation.

    OUTPUT RULES:

    Return ONLY a valid JSON object.
    Include exactly these ten keys:
    Happy, Sad, Angry, Frustrated, Bored, Excited,
    Jealous, Insecure, Relieved, Neutral.

    Every value must be an integer from 0 to 100.
    Do not include explanations, comments, or Markdown code fences.

    Message:
    {message}
    """

    count += 1

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "qwen3:4b",
                "prompt": prompt,
                "stream": False,
                "format": "json",
                "think": False,
            },
            timeout=120
        )

        response.raise_for_status()
        result = response.json()["response"]
        emotion = json.loads(result)

        # Validate the returned emotion names.
        if not isinstance(emotion, dict):
            raise ValueError("Output is not a JSON object")

        if set(emotion.keys()) != set(emotion_names):
            raise ValueError("Missing or unexpected emotion names")

        # Validate every score before saving anything.
        for emotion_name in emotion_names:
            score = emotion[emotion_name]

            if type(score) is not int or not 0 <= score <= 100:
                raise ValueError(
                    f"Invalid score for {emotion_name}: {score}"
                )

        # Save only after the entire result passes validation.
        for emotion_name in emotion_names:
            df_text.loc[index, emotion_name] = emotion[emotion_name]

        success_count += 1

    except (json.JSONDecodeError, KeyError, ValueError) as error:
        failed_rows.append(index)
        print(f"Invalid output at row {index}: {error}")

    except requests.RequestException as error:
        failed_rows.append(index)
        print(f"Request failed at row {index}: {error}")

    print(
        f"Processed: {count}/{len(sample)} | "
        f"Successful: {success_count} | "
        f"Failed: {len(failed_rows)}"
    )

print("Batch finished!")
print(f"Successfully labeled: {success_count}")
print(f"Failed rows: {failed_rows}")


Processed: 1/100 | Successful: 1 | Failed: 0
Processed: 2/100 | Successful: 2 | Failed: 0
Processed: 3/100 | Successful: 3 | Failed: 0
Processed: 4/100 | Successful: 4 | Failed: 0
Processed: 5/100 | Successful: 5 | Failed: 0
Processed: 6/100 | Successful: 6 | Failed: 0
Processed: 7/100 | Successful: 7 | Failed: 0
Processed: 8/100 | Successful: 8 | Failed: 0
Processed: 9/100 | Successful: 9 | Failed: 0
Processed: 10/100 | Successful: 10 | Failed: 0
Processed: 11/100 | Successful: 11 | Failed: 0
Processed: 12/100 | Successful: 12 | Failed: 0
Processed: 13/100 | Successful: 13 | Failed: 0
Processed: 14/100 | Successful: 14 | Failed: 0
Processed: 15/100 | Successful: 15 | Failed: 0
Processed: 16/100 | Successful: 16 | Failed: 0
Processed: 17/100 | Successful: 17 | Failed: 0
Processed: 18/100 | Successful: 18 | Failed: 0
Processed: 19/100 | Successful: 19 | Failed: 0
Processed: 20/100 | Successful: 20 | Failed: 0
Processed: 21/100 | Successful: 21 | Failed: 0
Processed: 22/100 | Successful:

In [38]:
df_text.to_csv("emotion_dataset_labeled_test.csv", index=False)

In [36]:
df_text.sample(50)

,message,Happy,Sad,Angry,Frustrated,Bored,Excited,Jealous,Insecure,Relieved,Neutral
1199,I keep thinking if I looked better they would ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
804,I am considering replacing my current smartpho...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1280,I think I need to stay away from them for a wh...,0.0,20.0,0.0,30.0,0.0,0.0,40.0,0.0,0.0,20.0
987,I think I started liking one of the staff memb...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1492,I'm tired of being me.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1537,Maybe. I was already angry by then.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144,You know exactly what youre doing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
739,I thought today was going to be terrible,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
134,Sorry XD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1232,Part of me wants to expose both of them.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
emotion_columns = [
    "Happy", "Sad", "Angry", "Frustrated", "Bored",
    "Excited", "Jealous", "Insecure", "Relieved", "Neutral"
]

df_text.loc[df_text.index[:10], emotion_columns] = (
    df_text.loc[df_text.index[:10], emotion_columns].fillna(0)
)

In [44]:
df_text.loc[sample.index, emotion_columns].isna().sum()

Happy         2
Sad           2
Angry         2
Frustrated    2
Bored         2
Excited       2
Jealous       2
Insecure      2
Relieved      2
Neutral       2
dtype: int64

# Data cleaning part, stage 2

In [1]:
df

NameError: name 'df' is not defined